In [1]:
#!/usr/bin/env python
# coding: utf-8

# # Decoder Complexity Analysis for Composite DNA
# ## Parameter Count, Inference Throughput, and Training Time
# ## Across All Error Models (EZ17, G15, O17)

# =============================================================================
# CELL 1: DEVICE CONFIGURATION
# =============================================================================
import os
import torch

DEVICE_ID = "3"
os.environ["CUDA_VISIBLE_DEVICES"] = DEVICE_ID


# =============================================================================
# CELL 2: IMPORTS
# =============================================================================
import random
import pickle
import numpy as np
import time
from datetime import datetime
from collections import defaultdict
import json

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, random_split

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✅ Using device: {device}")
if torch.cuda.is_available():
    print(f"   GPU: {torch.cuda.get_device_name(0)}")


# =============================================================================
# CELL 3: CONFIGURATION
# =============================================================================

# Error model specifications
ERROR_MODELS = {
    "erlich": {"seq_length": 136, "name": "EZ17"},
    "grass": {"seq_length": 104, "name": "G15"},
    "organick": {"seq_length": 77, "name": "O17"}
}

# Alphabet configurations
ALPHABET_CONFIGS = {
    "A6": {"vocab_size": 10, "alphabet_mode": "2mix_only", "name": "$\\mathcal{A}_6$"},
    "A10": {"vocab_size": 14, "alphabet_mode": "2mix_3mix", "name": "$\\mathcal{A}_{10}$"},
    "A11": {"vocab_size": 15, "alphabet_mode": "2mix_3mix_4mix", "name": "$\\mathcal{A}_{11}$"},
    "A_eta": {"vocab_size": 34, "alphabet_mode": "eta0.2", "name": "$\\mathcal{A}_{0.2}$"}
}

# Model architecture parameters (same as training)
MODEL_CONFIG = {
    "input_channels": 4,
    "hidden_dim": 128,
    "num_layers": 2,
    "dropout": 0.2,
    "bidirectional": True,
}

# Benchmarking parameters
BENCHMARK_CONFIG = {
    "num_warmup_runs": 10,
    "num_benchmark_runs": 100,
    "batch_sizes": [1, 32, 64, 128, 256, 500],
    "default_batch_size": 500,
}

# Dataset parameters for training time estimation
DATASET_CONFIG = {
    "num_samples": 100000,
    "train_ratio": 0.8,
    "max_coverage": 25,
}

# Results directory
RESULTS_DIR = "./complexity_analysis_results"
os.makedirs(RESULTS_DIR, exist_ok=True)

print(f"{'='*60}")
print(f"📋 COMPLEXITY ANALYSIS CONFIGURATION")
print(f"{'='*60}")
print(f"   Error Models: {list(ERROR_MODELS.keys())}")
print(f"   Alphabets: {list(ALPHABET_CONFIGS.keys())}")
print(f"   Model: Bi-LSTM, hidden_dim={MODEL_CONFIG['hidden_dim']}, layers={MODEL_CONFIG['num_layers']}")
print(f"   Results directory: {RESULTS_DIR}")
print(f"{'='*60}")


# =============================================================================
# CELL 4: SEED FOR REPRODUCIBILITY
# =============================================================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)
print(f"🎲 Random seed set to: 42")


# =============================================================================
# CELL 5: MODEL DEFINITION (Same as training code)
# =============================================================================

class CompositeDecoderLSTM(nn.Module):
    """Bidirectional LSTM Decoder for Composite DNA."""
    def __init__(self, config):
        super(CompositeDecoderLSTM, self).__init__()
        
        self.lstm = nn.LSTM(
            input_size=config['input_channels'],
            hidden_size=config['hidden_dim'],
            num_layers=config['num_layers'],
            batch_first=True,
            bidirectional=config['bidirectional'],
            dropout=config['dropout'] if config['num_layers'] > 1 else 0
        )
        
        fc_in = config['hidden_dim'] * 2 if config['bidirectional'] else config['hidden_dim']
        self.fc = nn.Linear(fc_in, config['vocab_size'])
        
    def forward(self, x):
        x = x.permute(0, 2, 1)  # (Batch, 4, L) -> (Batch, L, 4)
        out, _ = self.lstm(x)
        logits = self.fc(out)
        return logits.permute(0, 2, 1)  # (Batch, vocab_size, L)


# =============================================================================
# CELL 6: BASELINE DECODERS (Copy from training code)
# =============================================================================

def min_distance_decoder(obs, ideal_vectors):
    """Minimum Euclidean Distance Decoder."""
    dists = torch.sum((obs.unsqueeze(2) - ideal_vectors.unsqueeze(0).unsqueeze(0)) ** 2, dim=3)
    return torch.argmin(dists, dim=2)


def kl_divergence_decoder(obs, ideal_vectors, epsilon=0.01):
    """KL Divergence Decoder."""
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    cross_entropy = -(obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmin(cross_entropy, dim=-1)


def maximum_likelihood_decoder(obs, ideal_vectors, epsilon=0.01):
    """Maximum Likelihood Decoder."""
    ideal_safe = ideal_vectors.clone()
    ideal_safe = torch.clamp(ideal_safe, min=epsilon)
    ideal_safe = ideal_safe / ideal_safe.sum(dim=-1, keepdim=True)
    
    obs_expanded = obs.unsqueeze(2)
    log_ideal = torch.log(ideal_safe).unsqueeze(0).unsqueeze(0)
    log_likelihood = (obs_expanded * log_ideal).sum(dim=-1)
    return torch.argmax(log_likelihood, dim=-1)


# =============================================================================
# CELL 7: IDEAL VECTORS BUILDERS (Copy from training code)
# =============================================================================

def build_ideal_vectors_A6():
    return torch.tensor([
        [1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0],
        [0.5, 0.0, 0.0, 0.5], [0.0, 0.5, 0.5, 0.0],
        [0.0, 0.5, 0.0, 0.5], [0.0, 0.0, 0.5, 0.5],
        [0.5, 0.5, 0.0, 0.0], [0.5, 0.0, 0.5, 0.0],
    ], dtype=torch.float32)


def build_ideal_vectors_A10():
    third = 1.0 / 3.0
    base = build_ideal_vectors_A6().tolist()
    base.extend([
        [third, third, third, 0.0], [third, third, 0.0, third],
        [third, 0.0, third, third], [0.0, third, third, third],
    ])
    return torch.tensor(base, dtype=torch.float32)


def build_ideal_vectors_A11():
    base = build_ideal_vectors_A10().tolist()
    base.append([0.25, 0.25, 0.25, 0.25])
    return torch.tensor(base, dtype=torch.float32)


def build_ideal_vectors_eta(eta=0.2, ell_values=[-2, -1, 0, 1, 2]):
    ideal_vectors = [
        [1.0, 0.0, 0.0, 0.0], [0.0, 1.0, 0.0, 0.0],
        [0.0, 0.0, 1.0, 0.0], [0.0, 0.0, 0.0, 1.0],
    ]
    pair_indices = [(0, 1), (0, 2), (0, 3), (1, 2), (1, 3), (2, 3)]
    for idx1, idx2 in pair_indices:
        for ell in ell_values:
            prob1 = 0.5 + ell * eta
            prob2 = 0.5 - ell * eta
            vec = [0.0, 0.0, 0.0, 0.0]
            vec[idx1] = prob1
            vec[idx2] = prob2
            ideal_vectors.append(vec)
    return torch.tensor(ideal_vectors, dtype=torch.float32)


def get_ideal_vectors(alphabet_key):
    if alphabet_key == "A6":
        return build_ideal_vectors_A6()
    elif alphabet_key == "A10":
        return build_ideal_vectors_A10()
    elif alphabet_key == "A11":
        return build_ideal_vectors_A11()
    elif alphabet_key == "A_eta":
        return build_ideal_vectors_eta()
    else:
        raise ValueError(f"Unknown alphabet: {alphabet_key}")


# =============================================================================
# CELL 8: PARAMETER COUNTING FUNCTIONS
# =============================================================================

def count_parameters(model):
    """Count total and trainable parameters."""
    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total_params, trainable_params


def count_parameters_by_layer(model):
    """Count parameters by layer type."""
    param_dict = {}
    for name, module in model.named_modules():
        if len(list(module.children())) == 0:
            params = sum(p.numel() for p in module.parameters())
            if params > 0:
                param_dict[name] = params
    return param_dict


def get_lstm_fc_breakdown(model):
    """Get LSTM and FC layer parameter breakdown."""
    lstm_params = sum(p.numel() for p in model.lstm.parameters())
    fc_params = sum(p.numel() for p in model.fc.parameters())
    return lstm_params, fc_params


def analyze_model_parameters(vocab_size, model_config):
    """Analyze parameters for a model with given vocab size."""
    config = model_config.copy()
    config['vocab_size'] = vocab_size
    
    model = CompositeDecoderLSTM(config)
    total, trainable = count_parameters(model)
    layer_params = count_parameters_by_layer(model)
    lstm_params, fc_params = get_lstm_fc_breakdown(model)
    
    return {
        'total_params': total,
        'trainable_params': trainable,
        'layer_breakdown': layer_params,
        'lstm_params': lstm_params,
        'fc_params': fc_params,
        'model': model
    }


# =============================================================================
# CELL 9: INFERENCE THROUGHPUT MEASUREMENT
# =============================================================================

def generate_random_input(batch_size, seq_length, device):
    """Generate random frequency matrix input."""
    raw = torch.rand(batch_size, 4, seq_length, device=device)
    normalized = raw / raw.sum(dim=1, keepdim=True)
    return normalized


def measure_lstm_throughput(model, batch_size, seq_length, num_warmup, num_runs, device):
    """Measure Bi-LSTM inference throughput."""
    model.eval()
    model.to(device)
    
    x = generate_random_input(batch_size, seq_length, device)
    
    # Warmup
    with torch.no_grad():
        for _ in range(num_warmup):
            _ = model(x)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    # Benchmark
    start_time = time.perf_counter()
    with torch.no_grad():
        for _ in range(num_runs):
            outputs = model(x)
            if device.type == 'cuda':
                torch.cuda.synchronize()
    end_time = time.perf_counter()
    
    total_time = end_time - start_time
    total_symbols = batch_size * seq_length * num_runs
    throughput = total_symbols / total_time
    latency_per_batch = (total_time / num_runs) * 1000
    
    return {
        'throughput_symbols_per_sec': throughput,
        'latency_ms_per_batch': latency_per_batch,
        'total_time_sec': total_time,
        'num_runs': num_runs
    }


def measure_baseline_throughput(decoder_func, ideal_vectors, batch_size, seq_length,
                                 num_warmup, num_runs, device):
    """Measure baseline decoder throughput."""
    ideal_vectors = ideal_vectors.to(device)
    
    raw = torch.rand(batch_size, seq_length, 4, device=device)
    obs = raw / raw.sum(dim=2, keepdim=True)
    
    # Warmup
    for _ in range(num_warmup):
        _ = decoder_func(obs, ideal_vectors)
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    # Benchmark
    start_time = time.perf_counter()
    for _ in range(num_runs):
        _ = decoder_func(obs, ideal_vectors)
        if device.type == 'cuda':
            torch.cuda.synchronize()
    end_time = time.perf_counter()
    
    total_time = end_time - start_time
    total_symbols = batch_size * seq_length * num_runs
    throughput = total_symbols / total_time
    latency_per_batch = (total_time / num_runs) * 1000
    
    return {
        'throughput_symbols_per_sec': throughput,
        'latency_ms_per_batch': latency_per_batch,
        'total_time_sec': total_time,
        'num_runs': num_runs
    }


# =============================================================================
# CELL 10: TRAINING TIME ESTIMATION
# =============================================================================

def estimate_training_time_per_epoch(model, batch_size, num_samples, seq_length, device):
    """Estimate training time per epoch."""
    model.train()
    model.to(device)
    
    num_batches = num_samples // batch_size
    
    x = generate_random_input(batch_size, seq_length, device)
    labels = torch.randint(0, model.fc.out_features, (batch_size, seq_length), device=device)
    
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
    
    # Warmup
    for _ in range(5):
        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
    
    if device.type == 'cuda':
        torch.cuda.synchronize()
    
    # Measure
    num_measure_batches = min(20, num_batches)
    
    start_time = time.perf_counter()
    for _ in range(num_measure_batches):
        optimizer.zero_grad()
        outputs = model(x)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        if device.type == 'cuda':
            torch.cuda.synchronize()
    end_time = time.perf_counter()
    
    time_per_batch = (end_time - start_time) / num_measure_batches
    estimated_epoch_time = time_per_batch * num_batches
    
    return {
        'time_per_batch_sec': time_per_batch,
        'estimated_epoch_time_sec': estimated_epoch_time,
        'estimated_epoch_time_min': estimated_epoch_time / 60,
        'num_batches_per_epoch': num_batches
    }


# =============================================================================
# CELL 11: TIME COMPLEXITY ANALYSIS
# =============================================================================

def get_time_complexity_string(decoder_type, model_config=None):
    """Get Big-O time complexity string."""
    if decoder_type in ['min_distance', 'kl', 'ml']:
        return "$\\mathcal{O}(|\\mathcal{A}|)$"
    elif decoder_type == 'lstm':
        d_h = model_config['hidden_dim'] if model_config else 128
        return f"$\\mathcal{O}(d_h^2)$"  # d_h = {d_h}
    return "Unknown"


def estimate_lstm_flops_per_position(input_size, hidden_size, num_layers, bidirectional, vocab_size):
    """
    Estimate FLOPs per sequence position for Bi-LSTM.
    
    LSTM cell operations per timestep:
    - 4 gates × (input_size × hidden_size + hidden_size × hidden_size) multiplications
    - Plus additions and activations
    
    Approximately: 8 × hidden_size × (input_size + hidden_size) per layer per direction
    """
    directions = 2 if bidirectional else 1
    
    # First layer
    flops_layer1 = 8 * hidden_size * (input_size + hidden_size) * directions
    
    # Subsequent layers (input is hidden_size * directions)
    input_size_subsequent = hidden_size * directions
    flops_per_subsequent_layer = 8 * hidden_size * (input_size_subsequent + hidden_size) * directions
    flops_other_layers = flops_per_subsequent_layer * (num_layers - 1)
    
    # FC layer: 2 × input_features × output_features (multiply-add)
    fc_input = hidden_size * directions
    flops_fc = 2 * fc_input * vocab_size
    
    total_flops_per_position = flops_layer1 + flops_other_layers + flops_fc
    
    return total_flops_per_position


def estimate_baseline_flops_per_position(vocab_size, decoder_type):
    """
    Estimate FLOPs per position for baseline decoders.
    
    Min Distance: For each symbol, compute L2 distance (4 subtractions, 4 squares, 3 adds)
                  Plus argmin over vocab_size values
                  ≈ 11 × vocab_size + vocab_size comparisons
    
    KL/ML: For each symbol, compute cross-entropy (4 multiplications, 4 logs, 3 adds)
           Plus argmin/argmax
           ≈ 11 × vocab_size + vocab_size comparisons (logs counted as ~1 FLOP)
    """
    if decoder_type == 'min_distance':
        # 4 subtract + 4 square + 3 add + 1 compare = ~12 per symbol
        return 12 * vocab_size
    else:  # KL or ML
        # 4 multiply + 4 log + 3 add + 1 compare = ~12 per symbol
        return 12 * vocab_size


# =============================================================================
# CELL 12: GPU MEMORY USAGE ESTIMATION
# =============================================================================

def estimate_model_memory(model, batch_size, seq_length, device):
    """Estimate GPU memory usage during inference."""
    model.eval()
    model.to(device)
    
    if device.type != 'cuda':
        return {'error': 'Memory measurement requires CUDA'}
    
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.empty_cache()
    
    x = generate_random_input(batch_size, seq_length, device)
    
    torch.cuda.synchronize()
    memory_before = torch.cuda.memory_allocated()
    
    with torch.no_grad():
        outputs = model(x)
        torch.cuda.synchronize()
    
    memory_after = torch.cuda.max_memory_allocated()
    
    model_memory = sum(p.numel() * p.element_size() for p in model.parameters())
    
    return {
        'inference_memory_mb': (memory_after - memory_before) / (1024 * 1024),
        'peak_memory_mb': memory_after / (1024 * 1024),
        'model_memory_mb': model_memory / (1024 * 1024)
    }


# =============================================================================
# CELL 13: MAIN ANALYSIS - PARAMETER COUNTING
# =============================================================================

print("\n" + "="*70)
print("📊 ANALYSIS 1: PARAMETER COUNTING")
print("="*70)
print("(Note: Parameters depend only on vocab_size, not error model)")

param_results = {}

for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
    vocab_size = alphabet_config['vocab_size']
    analysis = analyze_model_parameters(vocab_size, MODEL_CONFIG)
    param_results[alphabet_key] = analysis
    
    print(f"\n🔹 {alphabet_config['name']} (vocab_size={vocab_size}):")
    print(f"   Total Parameters: {analysis['total_params']:,}")
    print(f"   LSTM Parameters: {analysis['lstm_params']:,}")
    print(f"   FC Parameters: {analysis['fc_params']:,}")

# Summary table
print(f"\n{'='*70}")
print("📋 PARAMETER COUNT SUMMARY")
print(f"{'='*70}")
print(f"{'Decoder':<30} {'Total Params':>15} {'LSTM':>12} {'FC':>12}")
print(f"{'-'*69}")
print(f"{'Min. Distance':<30} {'0':>15} {'N/A':>12} {'N/A':>12}")
print(f"{'KL Divergence':<30} {'0':>15} {'N/A':>12} {'N/A':>12}")
print(f"{'Max. Likelihood':<30} {'0':>15} {'N/A':>12} {'N/A':>12}")
for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
    name = f"Bi-LSTM ({alphabet_config['name']})"
    total = param_results[alphabet_key]['total_params']
    lstm = param_results[alphabet_key]['lstm_params']
    fc = param_results[alphabet_key]['fc_params']
    print(f"{name:<30} {total:>15,} {lstm:>12,} {fc:>12,}")


# =============================================================================
# CELL 14: MAIN ANALYSIS - INFERENCE THROUGHPUT (ALL ERROR MODELS)
# =============================================================================

print("\n" + "="*70)
print("📊 ANALYSIS 2: INFERENCE THROUGHPUT (All Error Models)")
print("="*70)

throughput_results = defaultdict(lambda: defaultdict(dict))
batch_size = BENCHMARK_CONFIG['default_batch_size']
num_warmup = BENCHMARK_CONFIG['num_warmup_runs']
num_runs = BENCHMARK_CONFIG['num_benchmark_runs']

for error_key, error_config in ERROR_MODELS.items():
    seq_length = error_config['seq_length']
    print(f"\n{'='*50}")
    print(f"📍 Error Model: {error_config['name']} (seq_length={seq_length})")
    print(f"{'='*50}")
    
    for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
        print(f"\n   🔹 {alphabet_config['name']} (vocab_size={alphabet_config['vocab_size']}):")
        
        vocab_size = alphabet_config['vocab_size']
        ideal_vectors = get_ideal_vectors(alphabet_key)
        
        # Bi-LSTM
        config = MODEL_CONFIG.copy()
        config['vocab_size'] = vocab_size
        model = CompositeDecoderLSTM(config)
        
        lstm_result = measure_lstm_throughput(
            model, batch_size, seq_length, num_warmup, num_runs, device
        )
        throughput_results[error_key][alphabet_key]['Bi-LSTM'] = lstm_result
        print(f"      Bi-LSTM: {lstm_result['throughput_symbols_per_sec']:,.0f} symbols/sec")
        
        # Min Distance
        mindist_result = measure_baseline_throughput(
            min_distance_decoder, ideal_vectors, batch_size, seq_length,
            num_warmup, num_runs, device
        )
        throughput_results[error_key][alphabet_key]['Min_Distance'] = mindist_result
        print(f"      Min Distance: {mindist_result['throughput_symbols_per_sec']:,.0f} symbols/sec")
        
        # KL Divergence
        kl_result = measure_baseline_throughput(
            kl_divergence_decoder, ideal_vectors, batch_size, seq_length,
            num_warmup, num_runs, device
        )
        throughput_results[error_key][alphabet_key]['KL'] = kl_result
        print(f"      KL Divergence: {kl_result['throughput_symbols_per_sec']:,.0f} symbols/sec")
        
        # ML (same as KL typically)
        ml_result = measure_baseline_throughput(
            maximum_likelihood_decoder, ideal_vectors, batch_size, seq_length,
            num_warmup, num_runs, device
        )
        throughput_results[error_key][alphabet_key]['ML'] = ml_result
        print(f"      Max Likelihood: {ml_result['throughput_symbols_per_sec']:,.0f} symbols/sec")


# =============================================================================
# CELL 15: MAIN ANALYSIS - TRAINING TIME ESTIMATION (ALL ERROR MODELS)
# =============================================================================

print("\n" + "="*70)
print("📊 ANALYSIS 3: TRAINING TIME ESTIMATION (All Error Models)")
print("="*70)

training_results = defaultdict(dict)
num_samples = int(DATASET_CONFIG['num_samples'] * DATASET_CONFIG['train_ratio'])

for error_key, error_config in ERROR_MODELS.items():
    seq_length = error_config['seq_length']
    print(f"\n📍 Error Model: {error_config['name']} (seq_length={seq_length})")
    
    for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
        vocab_size = alphabet_config['vocab_size']
        config = MODEL_CONFIG.copy()
        config['vocab_size'] = vocab_size
        model = CompositeDecoderLSTM(config)
        
        train_result = estimate_training_time_per_epoch(
            model, batch_size, num_samples, seq_length, device
        )
        training_results[error_key][alphabet_key] = train_result
        
        print(f"   {alphabet_config['name']}: {train_result['estimated_epoch_time_min']:.2f} min/epoch")


# =============================================================================
# CELL 16: MAIN ANALYSIS - FLOPs PER POSITION
# =============================================================================

print("\n" + "="*70)
print("📊 ANALYSIS 4: FLOPs PER POSITION")
print("="*70)
print("(Note: FLOPs per position are independent of sequence length)")

flops_results = {}

for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
    vocab_size = alphabet_config['vocab_size']
    
    # LSTM FLOPs per position
    lstm_flops = estimate_lstm_flops_per_position(
        input_size=MODEL_CONFIG['input_channels'],
        hidden_size=MODEL_CONFIG['hidden_dim'],
        num_layers=MODEL_CONFIG['num_layers'],
        bidirectional=MODEL_CONFIG['bidirectional'],
        vocab_size=vocab_size
    )
    
    # Baseline FLOPs per position
    mindist_flops = estimate_baseline_flops_per_position(vocab_size, 'min_distance')
    kl_flops = estimate_baseline_flops_per_position(vocab_size, 'kl')
    
    flops_results[alphabet_key] = {
        'Bi-LSTM': lstm_flops,
        'Min_Distance': mindist_flops,
        'KL': kl_flops,
        'ML': kl_flops
    }
    
    print(f"\n🔹 {alphabet_config['name']} (vocab_size={vocab_size}):")
    print(f"   Bi-LSTM: {lstm_flops:,} FLOPs/position ({lstm_flops/1000:.1f}K)")
    print(f"   Min Distance: {mindist_flops:,} FLOPs/position")
    print(f"   KL/ML: {kl_flops:,} FLOPs/position")


# =============================================================================
# CELL 17: GPU MEMORY ANALYSIS
# =============================================================================

print("\n" + "="*70)
print("📊 ANALYSIS 5: GPU MEMORY USAGE")
print("="*70)

memory_results = defaultdict(dict)

if device.type == 'cuda':
    # Use EZ17 as reference for memory (longest sequences)
    ref_error = 'erlich'
    seq_length = ERROR_MODELS[ref_error]['seq_length']
    
    print(f"(Using {ERROR_MODELS[ref_error]['name']} seq_length={seq_length} as reference)")
    
    for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
        vocab_size = alphabet_config['vocab_size']
        config = MODEL_CONFIG.copy()
        config['vocab_size'] = vocab_size
        model = CompositeDecoderLSTM(config)
        
        mem_result = estimate_model_memory(model, batch_size, seq_length, device)
        memory_results[alphabet_key] = mem_result
        
        print(f"\n🔹 {alphabet_config['name']} (batch_size={batch_size}):")
        print(f"   Model Memory: {mem_result['model_memory_mb']:.2f} MB")
        print(f"   Inference Memory: {mem_result['inference_memory_mb']:.2f} MB")
        print(f"   Peak Memory: {mem_result['peak_memory_mb']:.2f} MB")
else:
    print("⚠️ GPU not available for memory measurement")


# =============================================================================
# CELL 18: GENERATE LATEX TABLE
# =============================================================================

print("\n" + "="*70)
print("📊 LATEX TABLE FOR PAPER")
print("="*70)

# Use EZ17 as representative for throughput (can note others in paper text)
ref_error = 'erlich'
ref_alphabet = 'A11'

latex_table = r"""
\begin{table}[t]
\caption{Computational complexity comparison of decoders}
\centering
\renewcommand{\arraystretch}{1.2}
\setlength{\tabcolsep}{3pt}
\begin{tabular}{l|c|c|c}
\toprule
\textbf{Decoder} & \textbf{Parameters} & \textbf{Time Complexity} & \textbf{Training} \\
\midrule
"""

# Baseline decoders
latex_table += f"Min.\\ Distance & 0 & $\\mathcal{{O}}(|\\mathcal{{A}}|)$ & None \\\\\n"
latex_table += f"KL Divergence & 0 & $\\mathcal{{O}}(|\\mathcal{{A}}|)$ & None \\\\\n"
latex_table += f"Max.\\ Likelihood & 0 & $\\mathcal{{O}}(|\\mathcal{{A}}|)$ & None \\\\\n"
latex_table += r"\midrule" + "\n"

# Bi-LSTM decoders
for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
    name = alphabet_config['name'].replace('$\\mathcal{', '').replace('}$', '').replace('\\', '')
    params = param_results[alphabet_key]['total_params']
    
    # Format parameters
    if params >= 1e6:
        params_str = f"$\\sim${params/1e6:.2f}M"
    else:
        params_str = f"$\\sim${params/1000:.0f}K"
    
    latex_table += f"Bi-LSTM ($\\mathcal{{{name}}}$) & {params_str} & $\\mathcal{{O}}(d_h^2)$ & Required \\\\\n"

latex_table += r"""
\bottomrule
\end{tabular}
\label{Table:Complexity}
\end{table}
"""

print(latex_table)


# =============================================================================
# CELL 19: DETAILED SUMMARY TABLES
# =============================================================================

print("\n" + "="*70)
print("📊 DETAILED SUMMARY TABLES")
print("="*70)

# Throughput Summary Table (Average across error models)
print("\n--- THROUGHPUT SUMMARY (symbols/sec) ---")
print(f"{'Alphabet':<12} {'Bi-LSTM':>15} {'Min Dist':>15} {'KL':>15} {'ML':>15}")
print(f"{'-'*72}")

for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
    # Average across error models
    lstm_avg = np.mean([throughput_results[ek][alphabet_key]['Bi-LSTM']['throughput_symbols_per_sec'] 
                        for ek in ERROR_MODELS.keys()])
    mindist_avg = np.mean([throughput_results[ek][alphabet_key]['Min_Distance']['throughput_symbols_per_sec'] 
                           for ek in ERROR_MODELS.keys()])
    kl_avg = np.mean([throughput_results[ek][alphabet_key]['KL']['throughput_symbols_per_sec'] 
                      for ek in ERROR_MODELS.keys()])
    ml_avg = np.mean([throughput_results[ek][alphabet_key]['ML']['throughput_symbols_per_sec'] 
                      for ek in ERROR_MODELS.keys()])
    
    name = alphabet_config['name'].replace('$\\mathcal{', '').replace('}$', '').replace('\\', '')
    print(f"{name:<12} {lstm_avg:>15,.0f} {mindist_avg:>15,.0f} {kl_avg:>15,.0f} {ml_avg:>15,.0f}")


# Training Time Summary
print("\n--- TRAINING TIME SUMMARY (minutes/epoch) ---")
print(f"{'Alphabet':<12}", end="")
for ek, ec in ERROR_MODELS.items():
    print(f" {ec['name']:>10}", end="")
print()
print(f"{'-'*42}")

for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
    name = alphabet_config['name'].replace('$\\mathcal{', '').replace('}$', '').replace('\\', '')
    print(f"{name:<12}", end="")
    for ek in ERROR_MODELS.keys():
        time_min = training_results[ek][alphabet_key]['estimated_epoch_time_min']
        print(f" {time_min:>10.2f}", end="")
    print()


# =============================================================================
# CELL 20: SAVE ALL RESULTS
# =============================================================================

print("\n" + "="*70)
print("💾 SAVING RESULTS")
print("="*70)

# Prepare serializable results
def convert_to_serializable(obj):
    """Convert numpy types to Python native types."""
    if isinstance(obj, np.integer):
        return int(obj)
    elif isinstance(obj, np.floating):
        return float(obj)
    elif isinstance(obj, np.ndarray):
        return obj.tolist()
    elif isinstance(obj, dict):
        return {k: convert_to_serializable(v) for k, v in obj.items()}
    elif isinstance(obj, list):
        return [convert_to_serializable(i) for i in obj]
    return obj


results_summary = {
    'timestamp': datetime.now().strftime("%Y-%m-%d %H:%M:%S"),
    'device': str(device),
    'gpu_name': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'N/A',
    'config': {
        'model': MODEL_CONFIG,
        'benchmark': BENCHMARK_CONFIG,
        'error_models': ERROR_MODELS,
        'alphabets': {k: {kk: vv for kk, vv in v.items() if kk != 'model'} 
                      for k, v in ALPHABET_CONFIGS.items()}
    },
    'parameter_counts': {
        ak: {
            'total': param_results[ak]['total_params'],
            'trainable': param_results[ak]['trainable_params'],
            'lstm': param_results[ak]['lstm_params'],
            'fc': param_results[ak]['fc_params'],
        } for ak in ALPHABET_CONFIGS.keys()
    },
    'throughput': convert_to_serializable(dict(throughput_results)),
    'training_time': convert_to_serializable(dict(training_results)),
    'flops_per_position': flops_results,
    'memory': convert_to_serializable(dict(memory_results)) if memory_results else 'N/A',
}

results_path = os.path.join(RESULTS_DIR, "complexity_analysis_all_models.json")
with open(results_path, 'w') as f:
    json.dump(results_summary, f, indent=4)

print(f"✅ Results saved to: {results_path}")


# =============================================================================
# CELL 21: FINAL PAPER-READY SUMMARY
# =============================================================================

print("\n" + "="*70)
print("📋 FINAL PAPER-READY SUMMARY")
print("="*70)

print("\n=== VALUES FOR TABLE IN PAPER ===\n")

print("Baseline Decoders (Min. Distance, KL Divergence, Max. Likelihood):")
print("  - Parameters: 0")
print("  - Time Complexity: O(|A|) per position")
print("  - Training: None")
print("  - Note: Only require ideal frequency vectors, no learned parameters")

print("\nBi-LSTM Decoders:")
for alphabet_key, alphabet_config in ALPHABET_CONFIGS.items():
    params = param_results[alphabet_key]['total_params']
    lstm_p = param_results[alphabet_key]['lstm_params']
    fc_p = param_results[alphabet_key]['fc_params']
    
    # Average throughput
    avg_tp = np.mean([
        throughput_results[ek][alphabet_key]['Bi-LSTM']['throughput_symbols_per_sec'] 
        for ek in ERROR_MODELS.keys()
    ])
    
    # Average training time
    avg_train = np.mean([
        training_results[ek][alphabet_key]['estimated_epoch_time_min'] 
        for ek in ERROR_MODELS.keys()
    ])
    
    print(f"\n  {alphabet_config['name']} (vocab_size={alphabet_config['vocab_size']}):")
    print(f"    - Total Parameters: {params:,} (~{params/1000:.0f}K)")
    print(f"      - LSTM: {lstm_p:,}, FC: {fc_p:,}")
    print(f"    - Time Complexity: O(d_h²) per position, d_h={MODEL_CONFIG['hidden_dim']}")
    print(f"    - Avg Throughput: {avg_tp:,.0f} symbols/sec ({avg_tp/1e6:.2f}M)")
    print(f"    - Avg Training Time: ~{avg_train:.1f} min/epoch")
    print(f"    - Training: Required")

print("\n" + "="*70)
print("✅ COMPLEXITY ANALYSIS COMPLETE")
print("="*70)

✅ Using device: cuda
   GPU: NVIDIA GeForce RTX 3080
📋 COMPLEXITY ANALYSIS CONFIGURATION
   Error Models: ['erlich', 'grass', 'organick']
   Alphabets: ['A6', 'A10', 'A11', 'A_eta']
   Model: Bi-LSTM, hidden_dim=128, layers=2
   Results directory: ./complexity_analysis_results
🎲 Random seed set to: 42

📊 ANALYSIS 1: PARAMETER COUNTING
(Note: Parameters depend only on vocab_size, not error model)

🔹 $\mathcal{A}_6$ (vocab_size=10):
   Total Parameters: 535,050
   LSTM Parameters: 532,480
   FC Parameters: 2,570

🔹 $\mathcal{A}_{10}$ (vocab_size=14):
   Total Parameters: 536,078
   LSTM Parameters: 532,480
   FC Parameters: 3,598

🔹 $\mathcal{A}_{11}$ (vocab_size=15):
   Total Parameters: 536,335
   LSTM Parameters: 532,480
   FC Parameters: 3,855

🔹 $\mathcal{A}_{0.2}$ (vocab_size=34):
   Total Parameters: 541,218
   LSTM Parameters: 532,480
   FC Parameters: 8,738

📋 PARAMETER COUNT SUMMARY
Decoder                           Total Params         LSTM           FC
-----------------------